# xHuBERT Experiment 4: Ablation Study
**De tai**: He thong goi y san pham dua tren phan tich giong noi va cam xuc
**Hoc vien**: Nguyen Tan Nhu | **GVHD**: TS. Bui Thanh Hung (IUH)

**Yeu cau**: `Runtime` -> `Change runtime type` -> **T4 GPU** -> Save

## Configurations
| Config | SLA | AP | Description |
|--------|-----|----|-------------|
| HuBERT-vanilla-FT | No | No | Fine-tune + last_hidden_state + mean-pool |
| xHuBERT-SLA | Yes | No | +Selective Layer Aggregation |
| xHuBERT-AP | No | Yes | +Attention Pooling |
| xHuBERT-full | Yes | Yes | Proposed method |

**Protocol**: LOSGO x 3 seeds x 6 folds = 72 runs (~72h total)

In [ ]:
# Kiem tra GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), "GB")
else:
    print("WARNING: GPU not enabled! Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/xhubert_results/"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

In [ ]:
# Install dependencies (Colab has torch, numpy, sklearn, matplotlib)
!pip install -q transformers==4.51.3 librosa huggingface-hub safetensors tqdm seaborn

# Verify versions
import torch, transformers, librosa
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"librosa:      {librosa.__version__}")

In [ ]:
# Upload .py modules to Colab
# Option 1: Upload manually via Files panel (drag & drop)
# Option 2: Clone from repo
# !git clone https://github.com/nhunet/xhubert-experiments.git
# %cd xhubert-experiments

# Verify required files
import os
required = [
    "config.py", "data.py", "features.py", "protocols.py",
    "stats.py", "utils.py",
    "models/__init__.py", "models/ml_classifiers.py",
    "models/xhubert.py", "models/hubert_vanilla.py",
    "models/fusion.py", "models/dl_1d.py", "models/dl_2d.py",
]
for f in required:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  [{status}]  {f}")

In [ ]:
# Download RAVDESS dataset
import os
RAVDESS_PATH = "./RAVDESS"
if not os.path.exists(RAVDESS_PATH):
    print("Downloading RAVDESS ...")
    !wget -q https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip
    !unzip -q Audio_Speech_Actors_01-24.zip -d RAVDESS/
    print("Done!")
else:
    print(f"RAVDESS already exists at {RAVDESS_PATH}")
    !find {RAVDESS_PATH} -name "*.wav" | wc -l

In [ ]:
# Set save directory
import config
config.SAVE_DIR = SAVE_DIR
config.CKPT_DIR = os.path.join(SAVE_DIR, "checkpoints")
os.makedirs(config.CKPT_DIR, exist_ok=True)
print(f"Results -> {config.SAVE_DIR}")
print(f"Checkpoints -> {config.CKPT_DIR}")

### Keep Colab Alive
Paste this into your **browser Console** (F12 -> Console) to prevent idle timeout:
```javascript
function ClickConnect() {
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
```

## Load Dataset

In [ ]:
from data import RavdessDataset
import config

dataset = RavdessDataset(sr=config.SR_HUBERT)
dataset.print_summary()

## Quick Test

In [ ]:
from experiments.exp4_ablation import run_exp4

df_quick = run_exp4(dataset=dataset, seeds=[42], configs=["HuBERT-vanilla-FT"],
                    force=True, quick=True)
print("Quick test passed!" if len(df_quick) > 0 else "FAILED!")

## Session 1: seed 42 (all 4 configs)

In [ ]:
df_s1 = run_exp4(dataset=dataset, seeds=[42], force=False)
print(df_s1.groupby("Model")["accuracy"].agg(["mean", "std"]).round(2))

## Session 2: seeds 43, 44

In [ ]:
df_s2 = run_exp4(dataset=dataset, seeds=[43, 44], force=False)
print(df_s2.groupby(["Model", "Seed"])["accuracy"].agg(["mean", "std"]).round(2))

## Statistical Tests

In [ ]:
import pandas as pd, numpy as np, os
from stats import paired_ttest, format_result

csv_path = os.path.join(config.SAVE_DIR, "results_exp4_ablation.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    # Average over seeds per fold
    avg = df.groupby(["Model", "Fold"])["accuracy"].mean().reset_index()

    vanilla = avg[avg["Model"] == "HuBERT-vanilla-FT"]["accuracy"].values
    full = avg[avg["Model"] == "xHuBERT-full"]["accuracy"].values

    if len(vanilla) == len(full) and len(vanilla) > 0:
        print(format_result("xHuBERT-full", "HuBERT-vanilla-FT", "Accuracy", full, vanilla))
else:
    print("Run ablation first!")

## Visualization

In [ ]:
from visualization.plots import plot_exp4_ablation
import pandas as pd, os

csv_path = os.path.join(config.SAVE_DIR, "results_exp4_ablation.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    plot_exp4_ablation(df)
    print("Figure saved!")
else:
    print("Run ablation first!")